# Kryptonite - Antarctic Sea-Ice Forecasting
## Production Pipeline: Real NSIDC Southern Hemisphere Data

| | |
|---|---|
| **Data** | 296 real NSIDC sic_pss25_*.nc files, Jan-Feb 2022 to 2026 |
| **Variable** | cdr_seaice_conc (uint8 0-100 percent, fill=255 for land) |
| **Cleaning** | mask(255->0) then divide by 100 -> [0.0, 1.0] |
| **Crop** | Raw [332,316] -> grid[100:228, 150:278] -> 128x128 (Maitri/Bharati) |
| **Windows** | T_in=5, T_out=3 cached as binary .pt files |
| **Architecture** | U-Net Encoder -> 2-Layer ConvLSTM -> U-Net Decoder |
| **Loss** | MSE (3x ice-edge) + 0.2*(1-SSIM) |
| **Optimizer** | AdamW lr=1e-4 + CosineAnnealingWarmRestarts |
| **Resilience** | Auto-resume last.ckpt, OOM batch-halving, MemoryGuardCallback |
| **Export** | torch.jit.script() -> model_final.pt -> Google Drive |

---


## Cell 1: Environment and Persistent Storage


In [ ]:
# ================================================================
#  CELL 1 — Environment & Persistent Storage
# ================================================================
import subprocess, sys, os

print("📦 Installing dependencies...")
pkgs = [
    "pytorch-lightning>=2.0",
    "torchmetrics>=1.0",
    "pytorch-msssim",
    "xarray",
    "netCDF4",
    "cftime",
]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
print("✓ Packages ready\n")

import torch
import pytorch_lightning as pl

# ── Directory layout ─────────────────────────────────────────
LOCAL_DATA   = "/content/local_dataset"
CACHE_DIR    = "/content/cached_tensors"
CKPT_ROOT    = "/content/drive/MyDrive/Kryptonite_Model_Checkpoints"
LOG_ROOT     = "/content/drive/MyDrive/Kryptonite_Logs"
ARTF_ROOT    = "/content/drive/MyDrive/Kryptonite_Artifacts"
for d in [LOCAL_DATA, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Mount Google Drive ───────────────────────────────────────
DRIVE_AVAILABLE = False
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_AVAILABLE = True
    for d in [CKPT_ROOT, LOG_ROOT, ARTF_ROOT]:
        os.makedirs(d, exist_ok=True)
    print("✓ Google Drive mounted")
except Exception as e:
    print(f"⚠ Drive unavailable ({e}) — using local fallback")
    CKPT_ROOT = "/content/checkpoints"
    LOG_ROOT  = "/content/logs"
    ARTF_ROOT = "/content/artifacts"
    for d in [CKPT_ROOT, LOG_ROOT, ARTF_ROOT]:
        os.makedirs(d, exist_ok=True)

LAST_CKPT = os.path.join(CKPT_ROOT, "last.ckpt")

# ── GPU verification ─────────────────────────────────────────
print("\n" + "="*55)
print("  HARDWARE VERIFICATION")
print("="*55)
if torch.cuda.is_available():
    gpu   = torch.cuda.get_device_properties(0)
    DEVICE = "cuda"
    print(f"  GPU    : {gpu.name}")
    print(f"  VRAM   : {gpu.total_memory/1e9:.1f} GB")
    print(f"  CUDA   : {torch.version.cuda}")
else:
    DEVICE = "cpu"
    print("  ⚠  GPU NOT AVAILABLE — running on CPU")
    print("     Runtime → Change runtime type → GPU → T4")
print(f"  PyTorch : {torch.__version__}")
print(f"  PL      : {pl.__version__}")
print("="*55)

print(f"\n  local_dataset : {LOCAL_DATA}")
print(f"  cached_tensors: {CACHE_DIR}")
print(f"  checkpoints   : {CKPT_ROOT}")
print(f"  last.ckpt     : {'EXISTS → will auto-resume' if os.path.isfile(LAST_CKPT) else 'not found → fresh start'}")
print("\n✓ Cell 1 complete")


## Cell 2: Real NSIDC Data Engineering (SeaIceDataModule)


In [ ]:
# ================================================================
#  CELL 2 — Real NSIDC Data Engineering (SeaIceDataModule)
#
#  Reads real sic_pss25_*.nc files from /content/local_dataset/
#  Variable: cdr_seaice_conc
#  Fill-value 255 = land / missing → masked to 0.0
#  Raw grid: [332, 316] Southern Hemisphere
#  Crop to Maitri/Bharati corridor: grid[100:228, 150:278] → 128x128
#  T_in=5, T_out=3  → sliding-window .pt cache
# ================================================================

import os, glob, shutil
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl

# ── Constants ──────────────────────────────────────────────
H, W     = 128, 128          # post-crop grid size
T_IN     = 5                 # input context days
T_OUT    = 3                 # forecast horizon days
CROP_R   = slice(100, 228)   # row crop on raw 332×316 grid
CROP_C   = slice(150, 278)   # col crop

# ── Step 1: Stage files from Drive → local NVMe ────────────
def stage_files_from_drive() -> list[str]:
    drive_glob = "/content/drive/MyDrive/SeaIce_Data/**/*.nc"
    remote     = sorted(glob.glob(drive_glob, recursive=True))
    if remote:
        print(f"  Found {len(remote)} .nc files in Drive — staging...")
        for src in remote:
            dst = os.path.join(LOCAL_DATA, os.path.basename(src))
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f"    Copied: {os.path.basename(src)}")
            else:
                print(f"    Already local: {os.path.basename(src)}")
    else:
        print("  No Drive files found — scanning local_dataset only")
    # Accept any .nc file — works with NSIDC CDR names (seaice_conc_daily_sh_*.nc)
    # as well as the PSS25 convention (sic_pss25_*.nc) without renaming.
    return sorted(glob.glob(os.path.join(LOCAL_DATA, "*.nc")))

# ── Step 2: Load & clean one NSIDC file ────────────────────
_first_file_logged = False

def load_nsidc_frame(nc_path: str) -> tuple[np.ndarray, str]:
    """
    Returns (frame_128x128 float32, date_str).

    CONFIRMED from real files:
      - Variable : cdr_seaice_conc
      - dtype    : uint8  (stored as 0–100 percent + fill=255 for land)
      - Shape    : (1, 332, 316)  Southern Hemisphere PSS25 grid

    Cleaning pipeline:
      1. Cast to float32
      2. Mask land/pole-hole (value==255) → 0.0
      3. Divide by 100.0  → [0.0, 1.0]
      4. Crop grid[100:228, 150:278] → 128×128 (Maitri/Bharati corridor)
    """
    global _first_file_logged
    import xarray as xr

    ds = xr.open_dataset(nc_path, mask_and_scale=False)

    SIC_VAR = "cdr_seaice_conc"
    if SIC_VAR not in ds:
        candidates = [v for v in ds.data_vars
                      if "seaice" in v.lower() or "sic" in v.lower()]
        if not candidates:
            raise KeyError(f"Cannot find SIC variable in {list(ds.data_vars)}")
        SIC_VAR = candidates[0]

    raw = ds[SIC_VAR].values.squeeze().astype(np.float32)  # (332, 316)

    # Log once to confirm real data format
    if not _first_file_logged:
        attrs = ds[SIC_VAR].attrs
        print(f"  [Data check] var={SIC_VAR}  shape={raw.shape}  "
              f"dtype=uint8  fill={attrs.get('_FillValue','?')}  "
              f"raw_range=[{int(raw.min())}, {int(raw.max())}]")
        print(f"  Cleaning: mask(255→0) then divide by 100 → [0.0, 1.0]")
        _first_file_logged = True

    # Step 1: mask land / pole-hole (255) → 0
    grid = np.where(raw == 255, 0.0, raw)

    # Step 2: normalize percent → fraction
    grid = grid / 100.0
    grid = np.clip(grid, 0.0, 1.0)

    # Step 3: spatial crop → 128×128
    frame = grid[CROP_R, CROP_C]

    # Date from filename: sic_pss25_YYYYMMDD_*.nc
    try:
        t = ds["time"].values[0]
        date_str = str(t)[:10]
    except Exception:
        base = os.path.basename(nc_path)
        # Extract 8-digit date from anywhere in filename
        import re
        m = re.search(r'(\d{8})', base)
        date_str = m.group(1) if m else base

    ds.close()
    return frame, date_str


# ── Step 3: Build and cache sliding-window tensors ──────────
def build_cache(frames: np.ndarray, dates: list, tag: str):
    """
    frames: (T, H, W) float32
    Builds N = T - T_IN - T_OUT + 1 windows.
    Saves {tag}_X.pt  (N, T_in, H, W)
    Saves {tag}_Y.pt  (N, T_out, H, W)
    """
    T = len(frames)
    if T < T_IN + T_OUT:
        raise ValueError(f"Not enough frames ({T}) for T_in+T_out={T_IN+T_OUT}")

    Xs, Ys = [], []
    for i in range(T - T_IN - T_OUT + 1):
        Xs.append(frames[i : i + T_IN])
        Ys.append(frames[i + T_IN : i + T_IN + T_OUT])

    X = torch.from_numpy(np.stack(Xs)).float()
    Y = torch.from_numpy(np.stack(Ys)).float()

    xp = os.path.join(CACHE_DIR, f"{tag}_X.pt")
    yp = os.path.join(CACHE_DIR, f"{tag}_Y.pt")
    torch.save(X, xp)
    torch.save(Y, yp)
    print(f"  Cached {tag}: X={list(X.shape)}  Y={list(Y.shape)}")
    return xp, yp

# ── Cached Tensor Dataset ───────────────────────────────────
class CachedSICDataset(Dataset):
    """Reads pre-cached .pt files — zero FUSE overhead at train time."""
    def __init__(self, x_path: str, y_path: str):
        self.X = torch.load(x_path, map_location="cpu")  # (N, T_in, H, W)
        self.Y = torch.load(y_path, map_location="cpu")  # (N, T_out, H, W)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return self.X[i].unsqueeze(1), self.Y[i].unsqueeze(1)
        # → (T_in, 1, H, W), (T_out, 1, H, W)

# ── DataModule ──────────────────────────────────────────────
class SeaIceDataModule(pl.LightningDataModule):
    def __init__(self, batch_size: int = 4, num_workers: int = 2):
        super().__init__()
        self.batch_size  = batch_size
        self.num_workers = num_workers
        self.pin         = DEVICE == "cuda"

    def setup(self, stage=None):
        self.train_ds = CachedSICDataset(
            os.path.join(CACHE_DIR, "train_X.pt"),
            os.path.join(CACHE_DIR, "train_Y.pt"))
        self.val_ds = CachedSICDataset(
            os.path.join(CACHE_DIR, "val_X.pt"),
            os.path.join(CACHE_DIR, "val_Y.pt"))
        self.test_ds = CachedSICDataset(
            os.path.join(CACHE_DIR, "test_X.pt"),
            os.path.join(CACHE_DIR, "test_Y.pt"))
        print(f"  DataModule setup: "
              f"train={len(self.train_ds)} "
              f"val={len(self.val_ds)} "
              f"test={len(self.test_ds)}")

    def _loader(self, ds, shuffle=False):
        return DataLoader(
            ds, batch_size=self.batch_size, shuffle=shuffle,
            num_workers=self.num_workers, pin_memory=self.pin,
            persistent_workers=(self.num_workers > 0), drop_last=shuffle)

    def train_dataloader(self): return self._loader(self.train_ds, shuffle=True)
    def val_dataloader(self):   return self._loader(self.val_ds)
    def test_dataloader(self):  return self._loader(self.test_ds)

# ── Execute ─────────────────────────────────────────────────
print("="*55)
print("  NSIDC DATA ENGINEERING")
print("="*55)

CACHE_TAGS = ["train", "val", "test"]
cache_ok   = all(
    os.path.isfile(os.path.join(CACHE_DIR, f"{t}_{x}.pt"))
    for t in CACHE_TAGS for x in ["X", "Y"]
)

if cache_ok:
    print("  ✓ Cache already exists — skipping rebuild")
else:
    print("  Building cache from real NSIDC files...")
    nc_files = stage_files_from_drive()

    real_loaded = False
    all_frames  = []
    all_dates   = []

    if nc_files:
        print(f"\n  Loading {len(nc_files)} NSIDC files:")
        try:
            for f in sorted(nc_files):
                frame, date = load_nsidc_frame(f)
                all_frames.append(frame)
                all_dates.append(date)
            all_frames = np.stack(all_frames)   # (T, 128, 128)
            real_loaded = True
            print(f"\n  ✓ Loaded {len(all_dates)} real frames")
            print(f"  Date range : {all_dates[0]}  →  {all_dates[-1]}")
            print(f"  Array shape: {all_frames.shape}")
            print(f"  Value range: [{all_frames.min():.4f}, {all_frames.max():.4f}]")
        except Exception as e:
            print(f"  ✗ Real data load FAILED: {e}")

    if not real_loaded:
        # ── SYNTHETIC FALLBACK ──────────────────────────────
        print("\n" + "!"*55)
        print("  [WARNING] REAL NSIDC FILES NOT LOADED.")
        print("  FALLING BACK TO SYNTHETIC DATA.")
        print("  Upload sic_pss25_*.nc to /content/local_dataset/")
        print("!"*55 + "\n")

        def _synth_frame(t, H, W, rng, offset=0.0):
            yy, xx = np.mgrid[0:H, 0:W]
            r = np.sqrt((yy-H//2)**2 + (xx-W//2)**2)
            ice_r = 40 + 12*np.sin((t/120.0)*2*np.pi + offset)
            g = np.clip(1.0 - (r - ice_r)/28.0, 0, 1)
            g[r < 12] = 0.0
            return (g + rng.normal(0, 0.04, (H, W))).clip(0, 1).astype(np.float32)

        rng = np.random.default_rng(42)
        all_frames = np.stack([_synth_frame(t, H, W, rng) for t in range(320)])
        all_dates  = [f"2022-{t:03d}" for t in range(320)]
        print(f"  Generated {len(all_frames)} synthetic frames")

    # Chronological 70/15/15 split
    N      = len(all_frames) - T_IN - T_OUT + 1
    n_tr   = int(0.70 * N)
    n_val  = int(0.15 * N)
    n_test = N - n_tr - n_val

    i_v    = n_tr  + T_IN + T_OUT - 1
    i_t    = i_v   + n_val

    build_cache(all_frames[:i_v],            all_dates[:i_v],          "train")
    build_cache(all_frames[n_tr:i_t],        all_dates[n_tr:i_t],     "val")
    build_cache(all_frames[n_tr+n_val:],     all_dates[n_tr+n_val:],  "test")
    print(f"  Split → train:{n_tr}  val:{n_val}  test:{n_test}")

BATCH_SIZE = 4
dm = SeaIceDataModule(batch_size=BATCH_SIZE, num_workers=2)
dm.setup()

print(f"\n  T_in={T_IN}  T_out={T_OUT}  H={H}  W={W}")
print(f"  batch_size={BATCH_SIZE}  pin_memory={DEVICE=='cuda'}")
print("\n✓ Cell 2 complete — DataModule ready")


## Cell 3: Hybrid Spatiotemporal Architecture


In [ ]:
# ================================================================
#  CELL 3 — Hybrid Spatiotemporal Architecture
#
#  U-Net Encoder → 2-Layer ConvLSTM Core → U-Net Decoder
#  Loss: MSE + SSIM
#  Optimizer: AdamW(lr=1e-4)
#
#  The model is designed to be torch.jit.script()-compatible:
#    - No dynamic Python constructs
#    - ConvLSTM loop uses explicit Optional[Tensor] state
#    - ModuleList iteration is supported by TorchScript
# ================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from torchmetrics import MetricCollection
from torchmetrics.classification import BinaryJaccardIndex, BinaryF1Score
from torchmetrics.regression import MeanSquaredError
from pytorch_msssim import ssim as compute_ssim

# ── U-Net Blocks ─────────────────────────────────────────────

class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class Down(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class Up(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        dh = skip.shape[2] - x.shape[2]
        dw = skip.shape[3] - x.shape[3]
        skip = skip[:, :, dh//2 : skip.shape[2] - dh//2,
                          dw//2 : skip.shape[3] - dw//2]
        return self.conv(torch.cat([skip, x], dim=1))

# ── ConvLSTM Cell ────────────────────────────────────────────

class ConvLSTMCell(nn.Module):
    """Standard ConvLSTM cell, TorchScript-compatible."""
    def __init__(self, in_ch: int, hidden: int, kernel: int = 3):
        super().__init__()
        self.hidden = hidden
        self.gates  = nn.Conv2d(in_ch + hidden, 4 * hidden, kernel,
                                padding=kernel // 2, bias=True)

    def forward(
        self,
        x: torch.Tensor,
        h: torch.Tensor,
        c: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        g = self.gates(torch.cat([x, h], dim=1))
        i, f, z, o = g.chunk(4, dim=1)
        c_ = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(z)
        h_ = torch.sigmoid(o) * torch.tanh(c_)
        return h_, c_

# ── 2-Layer ConvLSTM Core ────────────────────────────────────

class ConvLSTMCore(nn.Module):
    """
    2-layer ConvLSTM.
    Input:  (B, T, C, H, W)
    Output: (B, hidden, H, W)  — last hidden state of layer 2
    """
    def __init__(self, in_ch: int, hidden: int):
        super().__init__()
        self.cell1  = ConvLSTMCell(in_ch,  hidden)
        self.cell2  = ConvLSTMCell(hidden, hidden)
        self.hidden = hidden

    def forward(self, seq: torch.Tensor) -> torch.Tensor:
        B, T, C, Hh, Ww = seq.shape
        h1 = torch.zeros(B, self.hidden, Hh, Ww, device=seq.device)
        c1 = torch.zeros(B, self.hidden, Hh, Ww, device=seq.device)
        h2 = torch.zeros(B, self.hidden, Hh, Ww, device=seq.device)
        c2 = torch.zeros(B, self.hidden, Hh, Ww, device=seq.device)
        for t in range(T):
            h1, c1 = self.cell1(seq[:, t], h1, c1)
            h2, c2 = self.cell2(h1,        h2, c2)
        return h2   # (B, hidden, Hh, Ww)

# ── Hybrid UNet + ConvLSTM ───────────────────────────────────

class HybridSICNet(nn.Module):
    """
    Input:  (B, T_in, 1, H, W)
    Output: (B, T_out, 1, H, W)  in [0, 1]
    """
    def __init__(self, t_in: int = 5, t_out: int = 3, f: int = 32, lstm_h: int = 128):
        super().__init__()
        self.t_in  = t_in
        self.t_out = t_out

        # Shared encoder (applied to each input frame)
        self.enc1 = DoubleConv(1,    f)
        self.enc2 = Down(f,          f*2)
        self.enc3 = Down(f*2,        f*4)
        self.enc4 = Down(f*4,        f*8)
        self.bot  = Down(f*8,        f*16)     # (B, f*16, H/16, W/16)

        # 2-layer ConvLSTM over bottleneck sequence
        self.lstm  = ConvLSTMCore(f*16, lstm_h)

        # Project ConvLSTM output back to decoder channel width
        self.proj  = nn.Conv2d(lstm_h, f*16, 1, bias=False)

        # Decoder (shared weights, applied per forecast step)
        self.dec4  = Up(f*16, f*8)
        self.dec3  = Up(f*8,  f*4)
        self.dec2  = Up(f*4,  f*2)
        self.dec1  = Up(f*2,  f)

        # One prediction head per forecast step
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Conv2d(f, 1, 1), nn.Sigmoid())
            for _ in range(t_out)
        ])

        self._init()
        n = sum(p.numel() for p in self.parameters())
        print(f"  HybridSICNet  f={f}  lstm_h={lstm_h}  params={n/1e6:.2f}M")

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _encode(self, x_t: torch.Tensor):
        e1 = self.enc1(x_t)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b  = self.bot(e4)
        return b, e1, e2, e3, e4

    def _decode(self, ctx: torch.Tensor, e1, e2, e3, e4) -> torch.Tensor:
        d = self.dec4(ctx, e4)
        d = self.dec3(d,   e3)
        d = self.dec2(d,   e2)
        d = self.dec1(d,   e1)
        return d

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C, Hh, Ww = x.shape

        # Encode all input frames; keep last skip connections
        bots: list[torch.Tensor] = []
        e1 = e2 = e3 = e4 = x.new_zeros(1)   # placeholders
        for t in range(T):
            b, e1, e2, e3, e4 = self._encode(x[:, t])
            bots.append(b)

        # Run ConvLSTM over bottleneck sequence
        bot_seq = torch.stack(bots, dim=1)    # (B, T, f*16, H/16, W/16)
        ctx     = self.proj(self.lstm(bot_seq))   # (B, f*16, H/16, W/16)

        # Decode one step per forecast head
        preds: list[torch.Tensor] = []
        for head in self.heads:
            d   = self._decode(ctx, e1, e2, e3, e4)
            out = head(d)                         # (B, 1, H, W)
            preds.append(out)

        return torch.stack(preds, dim=1)          # (B, T_out, 1, H, W)

# ── Loss: MSE + SSIM ─────────────────────────────────────────

class SICLoss(nn.Module):
    """Weighted MSE (3× ice-edge) + 0.2*(1-SSIM)."""
    def __init__(self, ssim_w: float = 0.2, edge_w: float = 3.0):
        super().__init__()
        self.ssim_w = ssim_w
        self.edge_w = edge_w

    def forward(self, pred: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        # Edge emphasis: pixels in 10-25% SIC range
        edge    = ((tgt > 0.10) & (tgt < 0.25)).float()
        weights = 1.0 + (self.edge_w - 1.0) * edge
        mse     = (weights * (pred - tgt).pow(2)).sum() / weights.sum().clamp(1)

        # SSIM over flattened (B*T, 1, H, W)
        B, T, C, Hh, Ww = pred.shape
        p_f = pred.view(B * T, C, Hh, Ww).clamp(0, 1)
        t_f = tgt.view( B * T, C, Hh, Ww).clamp(0, 1)
        ssim_val = compute_ssim(p_f, t_f, data_range=1.0, size_average=True)

        return mse + self.ssim_w * (1.0 - ssim_val)

# ── LightningModule ──────────────────────────────────────────

class SeaIceForecastModel(pl.LightningModule):
    ICE_THR = 0.15   # 15% SIC = ice edge threshold

    def __init__(self, t_in: int = 5, t_out: int = 3,
                 f: int = 32, lstm_h: int = 128, lr: float = 1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.lr        = lr
        self.net       = HybridSICNet(t_in, t_out, f, lstm_h)
        self.criterion = SICLoss()

        # Binary metrics — expect integer 0/1 targets after thresholding
        kw = dict(threshold=self.ICE_THR)
        self.tr_iou  = BinaryJaccardIndex(**kw)
        self.tr_dice = BinaryF1Score(**kw)
        self.tr_mse  = MeanSquaredError()

        self.va_iou  = BinaryJaccardIndex(**kw)
        self.va_dice = BinaryF1Score(**kw)
        self.va_mse  = MeanSquaredError()

        self.te_iou  = BinaryJaccardIndex(**kw)
        self.te_dice = BinaryF1Score(**kw)
        self.te_mse  = MeanSquaredError()

        self._viz_batch = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    def _flat(self, t: torch.Tensor) -> torch.Tensor:
        return t.reshape(-1)

    def _bin(self, t: torch.Tensor) -> torch.Tensor:
        """Binarize continuous SIC to 0/1 ice mask using ICE_THR."""
        return (t.reshape(-1) >= self.ICE_THR).long()

    def _update_metrics(self, iou_m, dice_m, mse_m, p, y):
        """
        Binary metrics (IoU, Dice) need integer 0/1 targets.
        MSE metric keeps continuous float values.
        """
        p_flat = self._flat(p)
        y_flat = self._flat(y)
        p_bin  = self._bin(p)
        y_bin  = self._bin(y)
        iou_m.update(p_bin,  y_bin)
        dice_m.update(p_bin, y_bin)
        mse_m.update(p_flat, y_flat)

    def training_step(self, batch, _):
        x, y = batch
        p    = self(x)
        loss = self.criterion(p, y)
        self._update_metrics(self.tr_iou, self.tr_dice, self.tr_mse, p.detach(), y)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        self.log_dict({
            "tr_iou":  self.tr_iou.compute(),
            "tr_dice": self.tr_dice.compute(),
        })
        self.tr_iou.reset(); self.tr_dice.reset(); self.tr_mse.reset()

    def validation_step(self, batch, idx):
        x, y = batch
        p    = self(x)
        loss = self.criterion(p, y)
        self._update_metrics(self.va_iou, self.va_dice, self.va_mse, p, y)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        if idx == 0 and self._viz_batch is None:
            self._viz_batch = (x.detach().cpu(), y.detach().cpu(), p.detach().cpu())
        return loss

    def on_validation_epoch_end(self):
        self.log_dict({
            "va_iou":  self.va_iou.compute(),
            "va_dice": self.va_dice.compute(),
            "va_mse":  self.va_mse.compute(),
        }, prog_bar=True)
        self.va_iou.reset(); self.va_dice.reset(); self.va_mse.reset()

    def test_step(self, batch, _):
        x, y = batch
        p    = self(x)
        loss = self.criterion(p, y)
        self._update_metrics(self.te_iou, self.te_dice, self.te_mse, p, y)
        self.log("test_loss", loss)
        return loss

    def on_test_epoch_end(self):
        iou  = self.te_iou.compute()
        dice = self.te_dice.compute()
        mse  = self.te_mse.compute()
        self.log_dict({"te_iou": iou, "te_dice": dice, "te_mse": mse})
        self.te_iou.reset(); self.te_dice.reset(); self.te_mse.reset()
        print(f"\n  +-----------------------------+")
        print(f"  | HOLD-OUT TEST RESULTS       |")
        print(f"  | IoU  : {iou.item():.4f}              |")
        print(f"  | Dice : {dice.item():.4f}              |")
        print(f"  | MSE  : {mse.item():.6f}            |")
        print(f"  +-----------------------------+")

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-2)
        sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2)
        return {"optimizer": opt,
                "lr_scheduler": {"scheduler": sch, "interval": "epoch"}}


# ── Verify ───────────────────────────────────────────────────
print("="*55)
print("  ARCHITECTURE VERIFICATION")
print("="*55)

model = SeaIceForecastModel(t_in=T_IN, t_out=T_OUT, f=32, lstm_h=128, lr=1e-4)

with torch.no_grad():
    dummy  = torch.zeros(1, T_IN, 1, H, W)
    output = model(dummy)
    assert output.shape == (1, T_OUT, 1, H, W), f"Shape error: {output.shape}"
    assert output.min() >= 0.0 and output.max() <= 1.0, "Output out of [0,1]"

print(f"\n  Input  : {list(dummy.shape)}")
print(f"  Output : {list(output.shape)}")
print(f"  Range  : [{output.min().item():.4f}, {output.max().item():.4f}]")
print(f"  Loss   : MSE (3x edge emphasis) + 0.2*(1-SSIM)")
print(f"  Opt    : AdamW lr=1e-4 wd=1e-2 + CosineAnnealingWarmRestarts")
print("\n✓ Cell 3 complete — architecture ready")


## Cell 4: Resilient Multi-Hour Training Execution


In [ ]:
# ================================================================
#  CELL 4 — Resilient Multi-Hour Training Execution
#
#  - ModelCheckpoint → Google Drive
#  - Custom MemoryGuardCallback: gc + empty_cache every epoch
#  - Auto-Resume: checks last.ckpt before fit()
#  - OOM recovery: halves batch_size, doubles accumulate_grad_batches
# ================================================================

import gc, time, os
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    ModelCheckpoint, EarlyStopping, LearningRateMonitor, Callback
)

# ── Custom Memory Guard ───────────────────────────────────────

class MemoryGuardCallback(Callback):
    """
    Called at the end of every training epoch.
    Frees Python and CUDA caches to prevent multi-hour memory creep.
    """
    def on_train_epoch_end(self, trainer, pl_module):
        before = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        after = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
        if before > 0:
            print(f"  [MemGuard] ep={trainer.current_epoch+1}: "
                  f"freed {before-after:.2f} GB "
                  f"(before={before:.2f} after={after:.2f})")

# ── Callbacks ─────────────────────────────────────────────────

ckpt_cb = ModelCheckpoint(
    dirpath    = CKPT_ROOT,
    filename   = "seaice-ep{epoch:02d}-val{val_loss:.4f}",
    monitor    = "val_loss",
    mode       = "min",
    save_top_k = 3,
    save_last  = True,
    verbose    = True,
)
early_cb = EarlyStopping(
    monitor   = "val_loss",
    patience  = 10,
    mode      = "min",
    min_delta = 1e-5,
    verbose   = True,
)
lr_cb  = LearningRateMonitor(logging_interval="epoch")
mem_cb = MemoryGuardCallback()

# ── Auto-Resume Logic ─────────────────────────────────────────

def get_resume_path() -> str | None:
    """
    Checks /content/drive/MyDrive/Kryptonite_Model_Checkpoints/last.ckpt.
    Returns the path if it exists, else None (fresh training).
    """
    if os.path.isfile(LAST_CKPT):
        sz = os.path.getsize(LAST_CKPT) / 1e6
        print(f"  ✓ last.ckpt found ({sz:.1f} MB) — RESUMING from checkpoint")
        return LAST_CKPT
    print("  ○ No last.ckpt — starting fresh")
    return None

# ── OOM-safe Training Launcher ────────────────────────────────

def launch(model, dm, initial_bs=4, max_epochs=50, max_retries=4):
    bs    = initial_bs
    accum = 2
    ckpt_path = get_resume_path()

    for attempt in range(1, max_retries + 1):
        prec = "16-mixed" if DEVICE == "cuda" else "32-true"
        print(f"\n{'='*55}")
        print(f"  Attempt {attempt}  bs={bs}  accum={accum}  "
              f"eff_batch={bs*accum}  prec={prec}")
        print(f"{'='*55}")

        dm.batch_size = bs
        dm.setup()

        trainer = pl.Trainer(
            max_epochs              = max_epochs,
            accelerator             = "gpu" if DEVICE == "cuda" else "cpu",
            devices                 = 1,
            precision               = prec,
            accumulate_grad_batches = accum,
            gradient_clip_val       = 1.0,
            callbacks               = [ckpt_cb, early_cb, lr_cb, mem_cb],
            log_every_n_steps       = max(1, len(dm.train_dataloader()) // 4),
            enable_progress_bar     = True,
            enable_model_summary    = (attempt == 1),
            default_root_dir        = CKPT_ROOT,
        )

        try:
            trainer.fit(model, datamodule=dm, ckpt_path=ckpt_path)
            return trainer, model
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                # Check if last.ckpt was created during the failed run
                if os.path.isfile(LAST_CKPT):
                    ckpt_path = LAST_CKPT
                bs    = max(1, bs // 2)
                accum = accum * 2
                print(f"\n  ⚠ OOM — retrying with bs={bs} accum={accum}")
                time.sleep(3)
            else:
                raise

    raise RuntimeError("Training failed: OOM retries exhausted.")

# ── Launch ────────────────────────────────────────────────────

print("="*55)
print("  PRODUCTION TRAINING  (max_epochs=50)")
print("="*55)
print("  Checkpoint dir  :", CKPT_ROOT)
print("  Monitor         : val_loss (EarlyStopping patience=10)")
print("  Memory guard    : gc + empty_cache every epoch")

t0 = time.time()
final_trainer, trained_model = launch(model, dm, initial_bs=4, max_epochs=50)
wall = time.time() - t0

best_ckpt = ckpt_cb.best_model_path
print(f"\n  ✓ Training done   wall={wall/60:.1f} min")
print(f"  Best val_loss   : {ckpt_cb.best_model_score:.6f}")
print(f"  Best checkpoint : {best_ckpt}")
print(f"  Epochs run      : {final_trainer.current_epoch + 1}")
print("\n  ⚠ Reminder: if Colab disconnects during training,")
print("    re-run this cell — auto-resume will load last.ckpt")
print("    from Google Drive and continue seamlessly.")
print("\n✓ Cell 4 complete")


## Cell 5: Handoff Verification and Export


In [ ]:
# ================================================================
#  CELL 5 — Handoff Verification & Export
#
#  1. Load best checkpoint from Drive
#  2. Run test() on hold-out set  (fresh minimal trainer, avoids RecursionError)
#  3. 3-column dark-themed plot:
#       Ground Truth (+24h/+48h/+72h) | Predictions | Error Residuals
#  4. torch.jit.script() → model_final.pt → Drive
# ================================================================

import sys, os, datetime
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Prevent rich/IPython formatter recursion during test()
sys.setrecursionlimit(10000)

print("="*55)
print("  CELL 5 — VERIFICATION & EXPORT")
print("="*55)

# ── 1. Load best checkpoint ───────────────────────────────────
print(f"\n  Loading best checkpoint:\n  {best_ckpt}")
best_model = SeaIceForecastModel.load_from_checkpoint(best_ckpt)
best_model.eval()
print("  ✓ Checkpoint loaded")

# ── 2. Hold-out test evaluation ───────────────────────────────
# Use a FRESH minimal trainer — reusing final_trainer causes RecursionError
# because its rich/progress-bar context is already closed after training.
print("\n  Running test() on hold-out set...")
test_trainer = pl.Trainer(
    accelerator        = "gpu" if DEVICE == "cuda" else "cpu",
    devices            = 1,
    enable_progress_bar= False,   # avoids rich recursion
    logger             = False,
    enable_model_summary = False,
)
test_results = test_trainer.test(best_model, datamodule=dm, verbose=False)
r = test_results[0] if test_results else {}
print(f"\n  Test IoU  : {r.get('te_iou',  r.get('test_iou',  'N/A'))}")
print(f"  Test Dice : {r.get('te_dice', r.get('test_dice', 'N/A'))}")
print(f"  Test MSE  : {r.get('te_mse',  r.get('test_mse',  'N/A'))}")


# ── 3. Fetch one validation sample ───────────────────────────
best_model.eval()
device = next(best_model.parameters()).device

# Try to reuse the viz batch stored during validation
if best_model._viz_batch is not None:
    x_np, y_np, p_np = [v.numpy() for v in best_model._viz_batch]
    # Shapes: (B, T_in/out, 1, H, W)  → use sample 0
    x_np = x_np[0]   # (T_in, 1, H, W)
    y_np = y_np[0]   # (T_out, 1, H, W)
    p_np = p_np[0]   # (T_out, 1, H, W)
else:
    # Fallback: grab one batch from the DataLoader
    sample = next(iter(dm.val_dataloader()))
    x_t, y_t = sample
    with torch.no_grad():
        p_t = best_model(x_t.to(device)).cpu()
    x_np = x_t[0].numpy()    # (T_in,  1, H, W)
    y_np = y_t[0].numpy()    # (T_out, 1, H, W)
    p_np = p_t[0].numpy()    # (T_out, 1, H, W)

step_labels = ["+24h", "+48h", "+72h"][:T_OUT]

# ── 4. 3-Column Plot ──────────────────────────────────────────
# Rows: one per forecast step (T_out rows)
# Cols: Ground Truth | Model Prediction | Error Residual

CMAP_ICE = "Blues_r"
CMAP_ERR = "hot"
BG       = "#0a0f1a"

fig = plt.figure(figsize=(13, 4.5 * T_OUT), facecolor=BG)
fig.suptitle(
    "Kryptonite Antarctic Sea-Ice Forecast  ·  Verification\n"
    "U-Net Encoder + 2-Layer ConvLSTM + U-Net Decoder",
    color="#e2e8f0", fontsize=13, fontweight="bold", y=1.01,
)
gs = gridspec.GridSpec(T_OUT, 3, figure=fig, hspace=0.08, wspace=0.06)

COL_TITLES  = ["Ground Truth", "ConvLSTM Prediction", "Error Residual  |Pred − GT|"]
COL_COLORS  = ["#34d399",     "#38bdf8",              "#fb923c"]

for row in range(T_OUT):
    gt  = y_np[row, 0]                          # (H, W)
    pr  = np.clip(p_np[row, 0], 0, 1)
    err = np.abs(pr - gt)

    for col, (frame, cmap, vmax, color) in enumerate([
        (gt,  CMAP_ICE, 1.0,  COL_COLORS[0]),
        (pr,  CMAP_ICE, 1.0,  COL_COLORS[1]),
        (err, CMAP_ERR, 0.35, COL_COLORS[2]),
    ]):
        ax = fig.add_subplot(gs[row, col])
        ax.set_facecolor(BG)
        ax.imshow(frame, cmap=cmap, vmin=0, vmax=vmax, origin="upper",
                  interpolation="nearest")
        # Ice-edge contour at 15% threshold
        ax.contour(frame, levels=[0.15], colors=["#ff6b00"],
                   linewidths=0.8, alpha=0.85)
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_edgecolor("#1e293b")

        # Row label
        if col == 0:
            ax.set_ylabel(step_labels[row], color="#94a3b8",
                          fontsize=11, labelpad=6)
        # Column title (top row only)
        if row == 0:
            ax.set_title(COL_TITLES[col], color=color,
                         fontsize=11, fontweight="bold", pad=8)
        # MSE annotation on error column
        if col == 2:
            mse_v = float(np.mean(err**2))
            ax.text(0.02, 0.97, f"MSE={mse_v:.5f}",
                    transform=ax.transAxes, fontsize=7, color="#fb923c",
                    va="top", ha="left",
                    bbox=dict(boxstyle="round,pad=0.2",
                              facecolor=BG, alpha=0.85))

# Shared colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.013, 0.70])
sm = plt.cm.ScalarMappable(cmap=CMAP_ICE, norm=plt.Normalize(0, 1))
cb = fig.colorbar(sm, cax=cbar_ax)
cb.set_label("Sea-Ice Concentration", color="#94a3b8", fontsize=9)
plt.setp(cb.ax.yaxis.get_ticklabels(), color="#94a3b8")

ts        = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
artf_path = os.path.join(ARTF_ROOT, f"verification_{ts}.png")
fig.savefig(artf_path, dpi=140, bbox_inches="tight",
            facecolor=BG, edgecolor="none")
plt.show()
print(f"\n  ✓ Verification figure saved → {artf_path}")

# ── 5. TorchScript Export ─────────────────────────────────────

class ScriptWrapper(nn.Module):
    """
    Thin wrapper exposing only the pure PyTorch net.
    No Lightning / torchmetrics dependencies — fully scriptable.
    """
    def __init__(self, net: HybridSICNet):
        super().__init__()
        self.net = net

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T_in, 1, H, W) float32 in [0, 1]
        Returns:
            (B, T_out, 1, H, W) float32 in [0, 1]
        """
        return self.net(x)

print("\n  Exporting TorchScript...")
wrapper = ScriptWrapper(best_model.net)
wrapper.eval()

# torch.jit.script() — static graph, no tracing required
scripted = torch.jit.script(wrapper)

# Verify scripted model
dummy = torch.zeros(1, T_IN, 1, H, W)
with torch.no_grad():
    out_s = scripted(dummy)
    assert out_s.shape == (1, T_OUT, 1, H, W)
    assert 0.0 <= out_s.min().item() and out_s.max().item() <= 1.0
print(f"  Script output : {list(out_s.shape)}  "
      f"range=[{out_s.min().item():.4f}, {out_s.max().item():.4f}]")

# Save to Drive
pt_path = os.path.join(CKPT_ROOT, "model_final.pt")
torch.jit.save(scripted, pt_path)
pt_mb = os.path.getsize(pt_path) / 1e6
print(f"  ✓ Saved → {pt_path}  ({pt_mb:.1f} MB)")

# Reload verify
reloaded = torch.jit.load(pt_path)
with torch.no_grad():
    out_r = reloaded(dummy)
    diff  = (out_r - out_s).abs().max().item()
assert diff < 1e-5, f"Reload mismatch: {diff}"
print(f"  ✓ Reload verified  max_diff={diff:.2e}")

# ── FastAPI integration hint ──────────────────────────────────
print("""
  ─────────────────────────────────────────────────────────
  FastAPI backend usage (predict.py):
  ─────────────────────────────────────────────────────────
  import torch

  _model = torch.jit.load("model_final.pt")
  _model.eval()

  @app.post("/api/forecast")
  async def forecast(body: ForecastRequest):
      x = torch.tensor(body.frames)  # (1, 5, 1, 128, 128)
      with torch.no_grad():
          pred = _model(x)           # (1, 3, 1, 128, 128)
      return {"concentration": pred.numpy().tolist()}
  ─────────────────────────────────────────────────────────
""")

print("="*55)
print("  PIPELINE COMPLETE")
print("="*55)
print(f"  Best checkpoint : {best_ckpt}")
print(f"  TorchScript     : {pt_path}  ({pt_mb:.1f} MB)")
print(f"  Artifact        : {artf_path}")
print(f"  Test IoU        : {r.get('te_iou', r.get('test_iou', 'N/A'))}")
print(f"  Test Dice       : {r.get('te_dice', r.get('test_dice', 'N/A'))}")
print("="*55)
print("\n✓ Cell 5 complete — download model_final.pt from Google Drive")
